## Audio Preprocessing and Clipping

In order for the data to be viable for training, it needs to have uniform dimensions. Larger bird classifiers use 5 second clips for training, but its been show that most bird calls can be captured in 2 second clips.

This notebook:
1. Decode -> Mono -> Resample.
2. Caps long recordings by selecting a 30s region.
3. Splits into consecutive 3s windows.
4. Uses a RMS gate to remove silent windows.
5. Runs an existing large bird classifier (BirdNET) as a teacher to label windows as: 
   - target **species** (keep)
   - **non_bird** (keep)
   - **wrong_bird** (drop)
6. Saves a number of species clips and non-bird clips.
7. Writes to manifests.

#### Ensure all dependencies are installed (requirements.txt) 

### Imports and Configs

In [1]:
import sys, platform
import math
import numpy as np
import librosa
import tempfile
import random
import os
import re
import tempfile
import contextlib
import io
import soundfile as sf
import pandas as pd
from birdnetlib.analyzer import Analyzer
from birdnetlib import Recording
from tqdm.auto import tqdm
from pathlib import Path

/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "config.py").exists():
    raise FileNotFoundError(f"Couldn't find src/config.py from {Path.cwd()}")

sys.path.insert(0, str(repo_root))

from src.config import CONFIG

cfg = CONFIG.preprocessing


### Paths and output folders


In [3]:
_data_dir_cfg = Path(CONFIG.paths.data_dir)
_data_dir_candidates = [
    (repo_root / _data_dir_cfg).resolve(),
    (Path.cwd() / _data_dir_cfg).resolve(),
    (repo_root / "src" / "dataset" / _data_dir_cfg).resolve(),
]

DATA_DIR = next((p for p in _data_dir_candidates if p.exists()), _data_dir_candidates[0])
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir
CLIPS_DIR = DATA_DIR / CONFIG.paths.clips_dir

OUT_SPECIES_DIR = CLIPS_DIR / "species"
OUT_NONBIRD_DIR = CLIPS_DIR / "non_bird"

for p in [RAW_DIR, MANIFEST_DIR, OUT_SPECIES_DIR, OUT_NONBIRD_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)


DATA_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data
RAW_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/raw
MANIFEST_DIR: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests


### Setup BirdNet teacher
Shoutout to BirdNet for making this really easy

In [4]:
analyzer = Analyzer()

if analyzer is None:
        raise RuntimeError("Teacher analyzer not initialized.")

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.


/Users/justyna-przy/Projects/ISE/FYP/.venv/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


### Utilities

__Root Mean Square (RMS) measures the average signal power over time.__

In [5]:
def rms_dbfs(x: np.ndarray) -> float:
    """Return RMS loudness in a dBFS-like scale for a mono waveform."""

    rms = float(np.sqrt(np.mean(np.square(x)) + cfg.eps))
    return float(20.0 * math.log10(rms + cfg.eps))


In [6]:
def build_window_start_times(region_len_s: float) -> list[float]:
    """Return window start times in seconds for a region."""

    start_times = []
    s = cfg.skip_first_s

    while s + cfg.clip_len_s <= region_len_s + 1e-9:
        start_times.append(float(s))
        s += cfg.stride_s

    return start_times


In [7]:
def load_audio_segment(path: Path, sr: int, offset_s: float, duration_s: float) -> np.ndarray:
    """Load a slice of audio and return it as a mono numpy array."""

    y, _ = librosa.load(str(path), sr=sr, mono=True, offset=float(offset_s), duration=float(duration_s))
    return y


In [8]:
def save_clip(y_16k: np.ndarray, out_dir: Path, xc_id: str, start_s: float, end_s: float) -> str:
    """Save a 16-bit PCM WAV clip and return its path."""

    start_ms = int(round(start_s * 1000))
    end_ms = int(round(end_s * 1000))
    fname = f"XC{xc_id}__s{start_ms}__e{end_ms}.wav"

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / fname
    sf.write(str(out_path), y_16k, cfg.sample_rate_model, subtype="PCM_16")
    return str(out_path)


In [9]:
def choose_best_region(path: Path, total_len_s: float) -> tuple[float, np.ndarray]:
    """Pick the most active region for long recordings.

    If the recording is shorter than the cap, return the full audio. Otherwise,
    slide a fixed window of length cfg.recording_cap_s every cfg.step_cap_s and
    choose the region with the highest RMS.
    """

    if total_len_s <= cfg.recording_cap_s + 1e-9:
        y = load_audio_segment(path, cfg.sample_rate_model, 0.0, total_len_s)
        return 0.0, y

    best_start = 0.0
    best_score = -float("inf")
    best_y = None
    max_start = max(0.0, total_len_s - cfg.recording_cap_s)

    for s in np.arange(0.0, max_start + 1e-9, cfg.step_cap_s):
        y = load_audio_segment(path, cfg.sample_rate_model, float(s), cfg.recording_cap_s)
        score = rms_dbfs(y)
        if score > best_score:
            best_start = float(s)
            best_score = score
            best_y = y

    if best_y is None:
        best_y = load_audio_segment(path, cfg.sample_rate_model, 0.0, min(total_len_s, cfg.recording_cap_s))
        best_start = 0.0

    return best_start, best_y


### Using the BirdNet Teacher
BirdNET is used as an offline teacher model to automatically validate and label candidate 3-second audio clips. After RMS-based energy gating removes silent windows, each remaining clip is resampled to 48 kHz and analyzed with BirdNET. Based on the top detection and its confidence, each clip is classified as species (target species detected with high confidence), non_bird (no bird detected), or drop (a bird detected but not the target species). Only a small, randomly selected subset of species and non_bird clips per recording is retained.

In [10]:
def teacher_analyze_window(y_16k: np.ndarray, target_sci_name: str) -> dict:
    """Run BirdNET on a 3s window and return a decision and detection metadata."""

    # BirdNET expects 48 kHz input audio.
    y_48k = librosa.resample(y_16k, orig_sr=cfg.sample_rate_model, target_sr=cfg.sample_rate_teacher)

    # Creating a temp file as input
    fd, tmp_path = tempfile.mkstemp(suffix=".wav")
    os.close(fd)

    try:
        sf.write(tmp_path, y_48k, cfg.sample_rate_teacher, subtype="PCM_16")

        rec = Recording(analyzer, tmp_path, min_conf=cfg.bird_conf_thr)
        with contextlib.redirect_stdout(io.StringIO()):
            rec.analyze()
        detections = rec.detections or []
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    # Pick the top detection by confidence for the decision.
    top = None
    max_conf = 0.0
    for d in detections:
        c = float(d.get("confidence", 0.0))
        if c > max_conf:
            max_conf, top = c, d

    top_sci = (top.get("scientific_name") if top else "")
    top_common = (top.get("common_name") if top else "")
    top_conf = float(top.get("confidence", 0.0)) if top else 0.0

    target_norm = target_sci_name.strip().lower()
    is_target = bool(top_sci) and (top_sci.strip().lower() == target_norm) and (top_conf >= cfg.species_conf_thr)

    if len(detections) == 0:
        decision = "non_bird"
    elif is_target:
        decision = "species"
    else:
        decision = "drop"

    return {
        "detections": detections,
        "top_sci": top_sci,
        "top_common": top_common,
        "top_conf": top_conf,
        "max_conf": float(max_conf),
        "decision": decision,
    }


### Running the Pipeline

In [11]:
def _resolve_local_source_path(local_path_value: str | Path) -> Path | None:
    """Resolve paths saved in manifests across OSes (Windows and macOS/Linux)."""
    raw = str(local_path_value).strip()
    if not raw:
        return None

    candidates: list[Path] = []
    for candidate_str in [raw, raw.replace("\\", "/")]:
        p = Path(candidate_str)
        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.extend([
                Path.cwd() / p,
                repo_root / p,
                DATA_DIR.parent / p,
                DATA_DIR / p,
            ])

    seen = set()
    for p in candidates:
        rp = p.resolve(strict=False)
        key = str(rp)
        if key in seen:
            continue
        seen.add(key)
        if rp.exists():
            return rp

    return None


def process_species(species_label: str, max_recordings: int | None = None) -> pd.DataFrame:
    """Process one species and write its clip manifest."""

    in_csv = MANIFEST_DIR / f"{species_label}_downloaded.csv"
    if not in_csv.exists():
        raise FileNotFoundError(f"Missing: {in_csv}")

    df = pd.read_csv(in_csv)
    if max_recordings is not None:
        df = df.head(max_recordings).copy()

    out_rows = []

    for _, r in tqdm(df.iterrows(), total=len(df), desc=species_label):
        xc_id = str(r.get("xc_id", ""))
        sci_name = str(r.get("sci_name", ""))

        src_path = _resolve_local_source_path(r.get("local_path", ""))
        if src_path is None:
            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(r.get("local_path", "")),
                "selected": 0,
                "final_class": "drop",
                "error": "missing_source",
            })
            continue

        try:
            total_len_s = float(librosa.get_duration(path=str(src_path)))
        except Exception:
            try:
                y_tmp, _ = librosa.load(str(src_path), sr=cfg.sample_rate_model, mono=True)
                total_len_s = float(len(y_tmp) / cfg.sample_rate_model)
            except Exception:
                out_rows.append({
                    "species_label": species_label,
                    "xc_id": xc_id,
                    "sci_name": sci_name,
                    "source_path": str(src_path),
                    "selected": 0,
                    "final_class": "drop",
                    "error": "load_failed",
                })
                continue

        try:
            region_start_s, y_region = choose_best_region(src_path, total_len_s)
        except Exception:
            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "selected": 0,
                "final_class": "drop",
                "error": "region_load_failed",
            })
            continue

        region_len_s = float(len(y_region) / cfg.sample_rate_model)

        starts = build_window_start_times(region_len_s)
        if not starts:
            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "selected": 0,
                "final_class": "drop",
                "error": "no_windows",
            })
            continue

        candidates = []
        for s in starts:
            start_i = int(round(s * cfg.sample_rate_model))
            end_i = start_i + int(round(cfg.clip_len_s * cfg.sample_rate_model))
            if end_i > len(y_region):
                continue
            w = y_region[start_i:end_i]
            candidates.append({
                "start_s": float(region_start_s + s),
                "end_s": float(region_start_s + s + cfg.clip_len_s),
                "rms_db": rms_dbfs(w),
                "wave_16k": w,
            })

        if not candidates:
            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "selected": 0,
                "final_class": "drop",
                "error": "no_candidates",
            })
            continue

        rms_vals = np.array([c["rms_db"] for c in candidates], dtype=float)
        thr = max(cfg.rms_abs_min_db, float(np.percentile(rms_vals, cfg.rms_keep_percentile)))
        gated = [c for c in candidates if c["rms_db"] >= thr]
        if not gated:
            gated = [candidates[int(np.argmax(rms_vals))]]

        for c in gated:
            t = teacher_analyze_window(c["wave_16k"], sci_name)
            c.update({
                "teacher_decision": t["decision"],
                "teacher_top_sci": t["top_sci"],
                "teacher_top_common": t["top_common"],
                "teacher_top_conf": t["top_conf"],
                "teacher_max_conf": t["max_conf"],
            })

        species_pos = [c for c in gated if c["teacher_decision"] == "species"]
        nonbird = [c for c in gated if c["teacher_decision"] == "non_bird"]

        try:
            seed_xc = int(float(xc_id))
        except Exception:
            seed_xc = abs(hash(xc_id)) % (2**31)
        rng = random.Random(cfg.seed + seed_xc)
        rng.shuffle(species_pos)
        sel_species = species_pos[:cfg.max_species_clips_per_rec]

        nonbird_sorted = sorted(nonbird, key=lambda x: x["teacher_max_conf"])
        sel_nonbird = nonbird_sorted[:cfg.nonbird_clips_per_rec]

        out_species_dir = OUT_SPECIES_DIR / species_label
        out_nonbird_dir = OUT_NONBIRD_DIR / species_label

        selected_set = set((c["start_s"], c["end_s"]) for c in (sel_species + sel_nonbird))

        for c in gated:
            key = (c["start_s"], c["end_s"])
            selected = int(key in selected_set)
            clip_path = ""
            selected_reason = ""

            if selected:
                if c in sel_species:
                    clip_path = save_clip(c["wave_16k"], out_species_dir, xc_id, c["start_s"], c["end_s"])
                    selected_reason = "rand_species"
                    final_class = "species"
                else:
                    clip_path = save_clip(c["wave_16k"], out_nonbird_dir, xc_id, c["start_s"], c["end_s"])
                    selected_reason = "nonbird_lowconf"
                    final_class = "non_bird"
            else:
                final_class = c["teacher_decision"]

            out_rows.append({
                "species_label": species_label,
                "xc_id": xc_id,
                "sci_name": sci_name,
                "source_path": str(src_path),
                "start_s": c["start_s"],
                "end_s": c["end_s"],
                "rms_db": c["rms_db"],
                "rms_gate_thr_db": thr,
                "teacher_decision": c["teacher_decision"],
                "teacher_top_sci": c.get("teacher_top_sci", ""),
                "teacher_top_common": c.get("teacher_top_common", ""),
                "teacher_top_conf": c.get("teacher_top_conf", 0.0),
                "teacher_max_conf": c.get("teacher_max_conf", 0.0),
                "final_class": final_class,
                "selected": selected,
                "selected_reason": selected_reason,
                "clip_path": clip_path,
                "error": "",
            })

    out_df = pd.DataFrame(out_rows)
    if "selected" not in out_df.columns:
        out_df["selected"] = 0
    if "final_class" not in out_df.columns:
        out_df["final_class"] = "drop"
    if "xc_id" not in out_df.columns:
        out_df["xc_id"] = ""

    out_csv = MANIFEST_DIR / f"{species_label}_clips.csv"
    out_df.to_csv(out_csv, index=False)
    print("Wrote:", out_csv)
    return out_df


In [12]:
def list_downloaded_species() -> list[str]:
    """Return species labels with downloaded manifests."""

    files = sorted(MANIFEST_DIR.glob("*_downloaded.csv"))
    species = []
    for f in files:
        m = re.match(r"(.+)_downloaded\.csv$", f.name)
        if m:
            species.append(m.group(1))
    return species


def process_all_species(max_species: int | None = None, max_recordings_per_species: int | None = None) -> pd.DataFrame:
    """Process all downloaded species and write a summary manifest."""

    species_list = list_downloaded_species()
    if max_species is not None:
        species_list = species_list[:max_species]

    summaries = []
    for sp in species_list:
        df_sp = process_species(sp, max_recordings=max_recordings_per_species)

        if "selected" in df_sp.columns:
            sel = df_sp[df_sp["selected"] == 1]
        else:
            sel = df_sp.iloc[0:0]

        selected_species = int((sel["final_class"] == "species").sum()) if "final_class" in sel.columns else 0
        selected_nonbird = int((sel["final_class"] == "non_bird").sum()) if "final_class" in sel.columns else 0
        unique_recordings = int(df_sp["xc_id"].nunique()) if "xc_id" in df_sp.columns else 0

        summaries.append({
            "species_label": sp,
            "selected_total": int(len(sel)),
            "selected_species": selected_species,
            "selected_nonbird": selected_nonbird,
            "unique_recordings": unique_recordings,
        })

    sum_df = pd.DataFrame(summaries).sort_values("species_label")
    sum_csv = MANIFEST_DIR / "clips_summary.csv"
    sum_df.to_csv(sum_csv, index=False)
    print("Wrote:", sum_csv)
    return sum_df


### Example run


In [13]:
summary = process_all_species(max_species=None, max_recordings_per_species=None)
summary


accipiter_nisus:   0%|          | 0/127 [00:00<?, ?it/s]

accipiter_nisus:  42%|████▏     | 53/127 [00:11<00:04, 14.84it/s]Note: Illegal Audio-MPEG-Header 0x69706974 at offset 219345.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
accipiter_nisus: 100%|██████████| 127/127 [00:16<00:00,  7.70it/s]


Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/accipiter_nisus_clips.csv


acrocephalus_schoenobaenus: 100%|██████████| 250/250 [00:29<00:00,  8.53it/s]


Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/acrocephalus_schoenobaenus_clips.csv


...

turdus_philomelos: 100%|██████████| 250/250 [00:23<00:00, 10.61it/s]


Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/turdus_philomelos_clips.csv


tyto_alba: 100%|██████████| 250/250 [00:12<00:00, 19.53it/s]

Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/tyto_alba_clips.csv
Wrote: /Users/justyna-przy/Projects/ISE/FYP/src/dataset/bird_data/manifests/clips_summary.csv


,species_label,selected_total,selected_species,selected_nonbird,unique_recordings
0,accipiter_nisus,187,157,30,127
1,acrocephalus_schoenobaenus,466,405,61,250
2,aegithalos_caudatus,526,504,22,250
3,alcedo_atthis,306,256,50,250
4,anthus_pratensis,347,293,54,250
5,apus_apus,438,418,20,250
6,buteo_buteo,403,372,31,250
7,carduelis_carduelis,540,490,50,250
8,chloris_chloris,467,403,64,250
9,cinclus_cinclus,250,232,18,146
